In [1]:
# !mkdir -p /content/data

In [2]:
# !tar -tzvf /content/ensemble_task_3_data_oct24-oct25.tar.gz

-rw-rw-r-- root/root 11185169388 2026-03-11 18:49 data.csv
-rw-rw-r-- root/root       44784 2026-03-11 18:49 devices.csv


In [3]:
!tar -xzvf /content/ensemble_task_3_data_oct24-oct25.tar.gz

data.csv
devices.csv


In [4]:

path_device = "devices.csv"
path_data = "data.csv"

In [5]:
import pandas as pd
from tqdm import tqdm

path_data =  "data.csv"

keep_columns = ['deviceId', 'timedate', 't1', 't2', 't7', 't8','t9', 'x1', 'x2', 'x3', 'deviceType']

all_sampled_chunks = []

reader = pd.read_csv(path_data, chunksize=1000000, usecols=keep_columns)

for chunk in tqdm(reader, desc="Slicing co 12 rekordów (co 1h)"):
    sampled = chunk.iloc[::12].copy()

    sampled['timedate'] = pd.to_datetime(sampled['timedate'])

    all_sampled_chunks.append(sampled)

df_data_sampled = pd.concat(all_sampled_chunks, ignore_index=True)

print(f"Pomyślnie wczytano {len(df_data_sampled)} rekordów.")

Slicing co 12 rekordów (co 1h): 65it [03:54,  3.61s/it]


Pomyślnie wczytane 5374819 rekordów.


In [6]:
df_data_sampled

,deviceId,timedate,t1,t2,t7,t8,t9,x1,x2,x3,deviceType
0,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 00:00:00+00:00,0.29,0.05,0.20,0.51,0.42,0.0,0.000000,8,19
1,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 01:00:00+00:00,0.29,0.05,0.20,0.54,0.40,0.0,0.579203,8,19
2,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 02:00:00+00:00,0.29,0.05,0.20,0.49,0.39,0.0,0.000000,8,19
3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 03:00:00+00:00,0.29,0.05,0.21,0.52,0.42,0.0,0.269766,8,19
4,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,2024-10-01 04:00:00+00:00,0.28,0.05,0.21,0.54,0.40,0.0,0.579203,8,19
...,...,...,...,...,...,...,...,...,...,...,...
5374814,ffd5e815486d2b094a0ff197111e96c998e11fda1626f7...,2025-10-31 19:00:00+00:00,0.29,0.05,0.21,0.48,0.35,0.0,NaN,8,11
5374815,ffd5e815486d2b094a0ff197111e96c998e11fda1626f7...,2025-10-31 20:00:00+00:00,0.29,0.05,0.21,0.47,0.35,0.0,NaN,8,11
5374816,ffd5e815486d2b094a0ff197111e96c998e11fda1626f7...,2025-10-31 21:00:00+00:00,0.29,0.05,0.21,0.47,0.35,0.0,NaN,8,11
5374817,ffd5e815486d2b094a0ff197111e96c998e11fda1626f7...,2025-10-31 22:00:00+00:00,0.29,0.05,0.21,0.46,0.35,0.0,NaN,8,11


In [7]:
df_data_sampled['timedate'] = pd.to_datetime(df_data_sampled['timedate'])

In [25]:
df_data_sampled['year'] = df_data_sampled['timedate'].dt.year
df_data_sampled['month'] = df_data_sampled['timedate'].dt.month
df_data_sampled['day'] = df_data_sampled['timedate'].dt.day
df_data_sampled['hour'] = df_data_sampled['timedate'].dt.hour

KeyError: 'timedate'

In [26]:
df_data_sampled.drop('timedate', axis=1, inplace=True)

KeyError: "['timedate'] not found in axis"

In [27]:
df_data_sampled['temp_delta'] = df_data_sampled['t2'] - df_data_sampled['t1']

df_data_sampled['deviceType'] = df_data_sampled['deviceType'].astype('category')
df_data_sampled['t1_nonlinear'] = np.exp(-df_data_sampled['t1'])

In [28]:
import numpy as np

df_data_sampled['month_sin'] = np.sin(2 * np.pi * df_data_sampled['month'] / 12)
df_data_sampled['month_cos'] = np.cos(2 * np.pi * df_data_sampled['month'] / 12)

df_data_sampled['hour_sin'] = np.sin(2 * np.pi * df_data_sampled['hour'] / 24)
df_data_sampled['hour_cos'] = np.cos(2 * np.pi * df_data_sampled['hour'] / 24)



In [29]:
df_data_sampled = df_data_sampled.sort_values(['deviceId', 'year', 'month', 'day', 'hour'])
df_data_sampled['x3'] = df_data_sampled['x3'].astype('category')
df_data_sampled['t1_rolling_3h'] = df_data_sampled.groupby('deviceId')['t1'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

In [30]:

train_df = df_data_sampled[
    (df_data_sampled['year'] == 2024) |
    ((df_data_sampled['year'] == 2025) & (df_data_sampled['month'] <= 4))
].copy()


predict_df = df_data_sampled[
    (df_data_sampled['year'] == 2025) & (df_data_sampled['month'] >= 5)
].copy()

In [31]:
import lightgbm as lgb
features_to_train = [
    't1', 't2', 't7', 't9', 'x1', 'x3',
    'deviceType', 'temp_delta',
    'month_sin', 'month_cos', 'hour_sin', 'hour_cos',
    't1_rolling_3h'
]

# Cel predykcyjny (target) [cite: 21, 28]
target = 'x2'

# Podział na zbiór treningowy i do predykcji
# Train: Październik 2024 - Kwiecień 2025
X_train = train_df[features_to_train]
y_train = train_df[target]

# Dane do prognozy: Maj 2025 - Październik 2025
X_predict = predict_df[features_to_train]

In [32]:


constraints = [-1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, -1]

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    monotone_constraints=constraints,
    categorical_feature=['deviceType', 'x3'], # Dodajemy x3 jako kategorię
    random_state=42
)

model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py:2137: UserWarning: categorical_feature keyword has been found in `params` and will be ignored.
Please use categorical_feature argument of the Dataset constructor to pass this parameter.
  _log_warning(
/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py:2159: UserWarning: categorical_feature in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")


[LightGBM] [Warning] categorical_feature is set=deviceType,x3, categorical_column=5,6 will be ignored. Current value: categorical_feature=deviceType,x3
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.133422 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 395
[LightGBM] [Info] Number of data points in the train set: 2887763, number of used features: 12
[LightGBM] [Info] Start training from score 0.157706


LGBMRegressor(categorical_feature=['deviceType', 'x3'], learning_rate=0.05,
              monotone_constraints=[-1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, -1],
              n_estimators=1000, random_state=42)

In [33]:
predict_df['prediction_raw'] = model.predict(X_predict)

submission = predict_df.groupby(['deviceId', 'year', 'month'])['prediction_raw'].mean().reset_index()

submission.rename(columns={'prediction_raw': 'prediction'}, inplace=True)

In [35]:
print(submission['month'].unique())

submission[['deviceId', 'year', 'month', 'prediction']].to_csv('submission_final.csv', index=False)

[ 5  6  7  8  9 10]
